In [3]:
!pip install pdfplumber python-docx striprtf pdf2image pytesseract

In [ ]:
import os
import time
from concurrent.futures import ProcessPoolExecutor, as_completed
from Italy.italy_scraping.ocr_utils import process_single_file

SOURCE_FOLDER = "documents"
OUTPUT_FOLDER = "txts"
# Lowering this slightly to 8 for better stability with your scrapers
MAX_WORKERS = 8 

def run_conversion():
    if not os.path.exists(OUTPUT_FOLDER):
        os.makedirs(OUTPUT_FOLDER)
    
    supported_extensions = ('.pdf', '.rtf', '.doc', '.docx')
    files_to_process = [f for f in os.listdir(SOURCE_FOLDER) if f.lower().endswith(supported_extensions)]
    total_files = len(files_to_process)
    
    print(f"🚀 Starting Parallel OCR with {MAX_WORKERS} workers for {total_files} files...\n")
    
    job_start_time = time.time()
    converted_count = 0
    skipped_count = 0
    error_count = 0

    with ProcessPoolExecutor(max_workers=MAX_WORKERS) as executor:
        # Submit all tasks
        future_to_file = {executor.submit(process_single_file, f): f for f in files_to_process}
        
        # as_completed(future_to_file) is more memory-efficient
        for index, future in enumerate(as_completed(future_to_file), 1):
            try:
                status, message, duration, was_ocr = future.result()
                
                progress_prefix = f"[{index}/{total_files}]"
                
                if status == "SKIP":
                    print(f"{progress_prefix} Skipping: {message} (TXT already exists)")
                    skipped_count += 1
                
                elif status == "SUCCESS":
                    print(f"{progress_prefix} Converting: {message}")
                    if was_ocr:
                        print(f"      [!] Scanned document detected. Running OCR...")
                    print(f"      [✓] Done in {duration:.2f}s")
                    converted_count += 1
                    
                elif status == "ERROR":
                    print(f"{progress_prefix} [X] Failed to convert: {message}")
                    error_count += 1
            except Exception as e:
                print(f"[{index}/{total_files}] [FATAL ERROR]: {e}")
                error_count += 1

    total_duration = time.time() - job_start_time
    print("\n" + "="*40)
    print(" JOB COMPLETE")
    print("="*40)
    print(f"Total processed:   {total_files}")
    print(f"Newly converted:   {converted_count}")
    print(f"Already existed:   {skipped_count}")
    print(f"Errors:            {error_count}")
    print(f"Total Time Taken:  {total_duration/60:.2f} minutes")
    print("="*40)

if __name__ == "__main__":
    run_conversion()

🚀 Starting Parallel OCR with 8 workers for 4800 files...

[1/4800] Skipping: Integrativo ESAARCO CCNL per i dipendenti delle piccole e medie industrie metalmeccaniche e di installazione di impianti (19142).pdf (TXT already exists)
[2/4800] Skipping: CCNL per gli addetti all'industria delle piastrelle di ceramica, dei materiali refrattari, ceramica sanitaria, di porcellana e ceramica per uso domestico e ornamentale, di ceramica tecnica, di tubi in gres (15979).doc (TXT already exists)
[3/4800] Skipping: CCNL micro, piccole e medie imprese esercenti l’attività nel settore del turismo e pubblici esercizi (18392).pdf (TXT already exists)
[4/4800] Skipping: CCNL per i dipendenti da strutture laboratoristiche e ambulatoriali private organizzate anche in forma cooperativistica (9804).doc (TXT already exists)
[5/4800] Skipping: CCNL Colf e Badanti (20120)_1.pdf (TXT already exists)
[6/4800] Skipping: MARITTIMI Medici di Bordo (6716)_1.rtf (TXT already exists)
[7/4800] Skipping: ATTIVITA' FUNE

In [1]:
import os
import shutil
import re
import pandas as pd

# --- CONFIGURATION ---
CSV_FILE = "master_ccnl_100_percent_complete.csv"
FINAL_FOLDER = "final_italy_txts"
LOOKUP_FOLDERS = ["txts", "documents"] # Where we look for existing files
SOURCE_FOLDERS = ["new_downloads", "pdfs"] # Where the original PDFs live

os.makedirs(FINAL_FOLDER, exist_ok=True)

def sanitize_title(title, code):
    clean = re.sub(r'[\\/*?:"<>|]', "", str(title))
    clean = " ".join(clean.split())[:150]
    return f"{code}_{clean}.txt"

def run_sync_and_cleanup():
    df = pd.read_csv(CSV_FILE)
    
    # Target rows where the Final TXT is blank
    missing_mask = df['Final_TXT_Filename'].isna() | (df['Final_TXT_Filename'] == "")
    missing_rows = df[missing_mask]
    
    print(f"🔎 Scanning for {len(missing_rows)} missing documents...")
    
    recovered = 0
    needs_redownload = []

    for idx, row in missing_rows.iterrows():
        acc_id = str(row['id accordo']).split('.')[0].strip()
        code = str(row['CCNL CNEL idfk']).strip()
        target_name = sanitize_title(row['Titolo'], code)
        
        # 1. SEARCH FOR EXISTING TXT (Recovery)
        found_existing = False
        for folder in LOOKUP_FOLDERS:
            if not os.path.exists(folder): continue
            for f in os.listdir(folder):
                if f"({acc_id})" in f and f.endswith(".txt"):
                    shutil.copy2(os.path.join(folder, f), os.path.join(FINAL_FOLDER, target_name))
                    df.at[idx, 'Final_TXT_Filename'] = target_name
                    recovered += 1
                    found_existing = True
                    break
            if found_existing: break
            
        if found_existing: continue

        # 2. IF NOT FOUND, MARK FOR REDOWNLOAD
        # We also check if the existing PDF is actually a fake (HTML error)
        needs_redownload.append(row['CCNL CNEL idfk'])

    # Save the updated master
    df.to_csv("master_ccnl_recovered_v2.csv", index=False, encoding="utf-8-sig")
    
    print("\n" + "="*40)
    print(f"✅ RECOVERED: {recovered} files found on disk.")
    print(f"❌ REMAINING: {len(needs_redownload)} files need redownload.")
    print("="*40)
    
    if needs_redownload:
        print("\n💡 ACTION REQUIRED:")
        print(f"Paste these codes into your scraper to get the missing files:")
        print(", ".join(list(set(needs_redownload))))

if __name__ == "__main__":
    run_sync_and_cleanup()

🔎 Scanning for 234 missing documents...

✅ RECOVERED: 0 files found on disk.
❌ REMAINING: 234 files need redownload.

💡 ACTION REQUIRED:
Paste these codes into your scraper to get the missing files:
A016, T511, S711, S333, K151, J121, H0C1, S223, S431, B011, A011, S611, IC91, D051, A101, S712, E015, S341, IC31, S714, S323, ID11, I134, S331, T111, A051, I491, F01A, S821, D013, H0B1, T141, D014, HV16, V100, H063, B142, G041, S013, E018, F018, I241, S412, S001, T013, S211, I012, J221, V112, S512, A031, B132, F058, G029, S111, J175, J131, T281, I112, S631, K700, H09C, I320, F065, H091, S612, S311, T912, J267, T611, E012, S003, IC50, D0L1, J126, E071, S622, S671, I481, H021, S233, J257, B254, E076, I531, K585, IB19, I140, H411, J342, E016, S421, T911, S103, H431, A060, H09D, I132, S321, IB17, V811, S513, A021, E031, S101
